In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [0]:
patient_readmission_features = spark.table("gold.patient_readmission_features").toPandas()

feature_cols = [
    "age",
    "encounter_count",
    "claim_count",
    "total_claim_amount",
    "avg_claim_amount",
    "condition_count",
    "gender",
    "age_group",
    "state",
    "is_deceased"
]

target_col = "readmitted_30_days"

In [0]:
df = patient_readmission_features[feature_cols + [target_col]].dropna(subset=[target_col])
df[target_col] = df[target_col].astype(int)
X = df[feature_cols]
y = df[target_col]

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [0]:
categorical_features = [
    "gender",
    "age_group",
    "state"
]
numeric_features = [
    "age",
    "encounter_count",
    "claim_count",
    "total_claim_amount",
    "avg_claim_amount",
    "condition_count",
    "is_deceased"
]

In [0]:
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [0]:
pipeline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=5000))
])

In [0]:
mlflow.set_experiment("/Shared/healthcare_readmission")

In [0]:
with mlflow.start_run():
    pipeline_model.fit(X_train, y_train)
    y_pred = pipeline_model.predict(X_test)
    y_prob = pipeline_model.predict_proba(X_test)[:, 1]
    mlflow.log_param("model_type", "logistic_regression")
    mlflow.log_param("feature_set", "scaled_numeric_plus_categorical")
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("precision", precision_score(y_test, y_pred, zero_division=0))
    mlflow.log_metric("recall", recall_score(y_test, y_pred, zero_division=0))
    mlflow.log_metric("f1", f1_score(y_test, y_pred, zero_division=0))
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_prob))
    mlflow.sklearn.log_model(pipeline_model, name="model")

    print("Accuracy:", accuracy_score(y_test, y_pred))

    print("Precision:", precision_score(y_test, y_pred, zero_division=0))

    print("Recall:", recall_score(y_test, y_pred, zero_division=0))

    print("F1:", f1_score(y_test, y_pred, zero_division=0))

    print("ROC AUC:", roc_auc_score(y_test, y_prob))

In [0]:
full_df = patient_readmission_features.copy()
score_df = full_df.dropna(subset=feature_cols)
all_prob = pipeline_model.predict_proba(score_df[feature_cols])[:, 1]
all_pred = pipeline_model.predict(score_df[feature_cols])

score_df["risk_score"] = all_prob

score_df["predicted_label"] = all_pred
pred_spark = spark.createDataFrame(score_df)
pred_spark.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.patient_readmission_predictions")

display(pred_spark.limit(1))

In [0]:
%sql
SELECT
    patient_key,
    risk_score,
    predicted_label,
    readmitted_30_days
FROM gold.patient_readmission_predictions
ORDER BY risk_score DESC
LIMIT 5;